In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV # Added GridSearchCV
import re                                       #regular expression library
from nltk.corpus import stopwords              #remove unncessary words like a,the,what etc
from nltk.stem.porter import PorterStemmer      #gives stem of a word
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [7]:
data=pd.read_csv('/content/FA-KES-Dataset.csv.zip', encoding='latin1')
data.head()

,unit_id,article_title,article_content,source,date,location,labels
0,1914947530,Syria attack symptoms consistent with nerve ag...,Wed 05 Apr 2017 Syria attack symptoms consiste...,nna,4/5/2017,idlib,0
1,1914947532,Homs governor says U.S. attack caused deaths b...,Fri 07 Apr 2017 at 0914 Homs governor says U.S...,nna,4/7/2017,homs,0
2,1914947533,Death toll from Aleppo bomb attack at least 112,Sun 16 Apr 2017 Death toll from Aleppo bomb at...,nna,4/16/2017,aleppo,0
3,1914947534,Aleppo bomb blast kills six Syrian state TV,Wed 19 Apr 2017 Aleppo bomb blast kills six Sy...,nna,4/19/2017,aleppo,0
4,1914947535,29 Syria Rebels Dead in Fighting for Key Alepp...,Sun 10 Jul 2016 29 Syria Rebels Dead in Fighti...,nna,7/10/2016,aleppo,0


In [8]:
import nltk
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [9]:
print(stopwords.words('english'))                             #givrs list of all stop words in eng language

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

DATA PREPROCESSING

In [10]:
data.shape

(804, 7)

In [11]:
data.describe()

,unit_id,labels
count,8.040000e+02,804.000000
mean,1.936024e+09,0.529851
std,1.876968e+07,0.499419
min,1.914948e+09,0.000000
25%,1.923848e+09,0.000000
50%,1.924058e+09,1.000000
75%,1.962496e+09,1.000000
max,1.965511e+09,1.000000


In [12]:
data.isnull().sum()

,0
unit_id,0
article_title,0
article_content,0
source,0
date,0
location,0
labels,0


In [13]:
data=data.fillna('')    #here no null value but in other cases  use this or drop the row itself

In [14]:
#merging article_title and	article_content	contents as we r going to use these for prediction
data['content']=data['article_title']+' '+data['location']
print(data['content'])

0      Syria attack symptoms consistent with nerve ag...
1      Homs governor says U.S. attack caused deaths b...
2      Death toll from Aleppo bomb attack at least 11...
3      Aleppo bomb blast kills six Syrian state TV al...
4      29 Syria Rebels Dead in Fighting for Key Alepp...
                             ...                        
799    Turkish Bombardment Kills 20 Civilians in Syri...
800    Martyrs as Terrorists Shell Aleppos Salah Eddi...
801    Chemical Attack Kills Five Syrians in Aleppo S...
802    5 Killed as Russian Military Chopper Shot down...
803    Syrian Army Kills 48 ISIL Terrorists in Deir E...
Name: content, Length: 804, dtype: object


In [15]:
Y=data["labels"]
X=data.drop(columns="labels",axis=1)        #this cell is not required jst for demo
print(X)
print(Y)

        unit_id                                      article_title  \
0    1914947530  Syria attack symptoms consistent with nerve ag...   
1    1914947532  Homs governor says U.S. attack caused deaths b...   
2    1914947533    Death toll from Aleppo bomb attack at least 112   
3    1914947534        Aleppo bomb blast kills six Syrian state TV   
4    1914947535  29 Syria Rebels Dead in Fighting for Key Alepp...   
..          ...                                                ...   
799  1965511221    Turkish Bombardment Kills 20 Civilians in Syria   
800  1965511222    Martyrs as Terrorists Shell Aleppos Salah Eddin   
801  1965511224  Chemical Attack Kills Five Syrians in Aleppo SANA   
802  1965511226  5 Killed as Russian Military Chopper Shot down...   
803  1965511231  Syrian Army Kills 48 ISIL Terrorists in Deir E...   

                                       article_content source       date  \
0    Wed 05 Apr 2017 Syria attack symptoms consiste...    nna   4/5/2017   
1    Fr

STEMMING: reducing to root word

In [16]:
port_stem=PorterStemmer()

In [17]:
def stemming(content):
  stemmed_content=re.sub('[^a-zA-Z]',' ',content)
  stemmed_content=stemmed_content.lower()
  stemmed_content=stemmed_content.split()
  stemmed_content=[port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
  stemmed_content=' '.join(stemmed_content)
  return stemmed_content


In [18]:
data['content']=data['content'].apply(stemming)
print(data['content'])

0      syria attack symptom consist nerv agent use idlib
1      hom governor say u attack caus death doesnt se...
2             death toll aleppo bomb attack least aleppo
3      aleppo bomb blast kill six syrian state tv aleppo
4          syria rebel dead fight key aleppo road aleppo
                             ...                        
799           turkish bombard kill civilian syria aleppo
800     martyr terrorist shell aleppo salah eddin aleppo
801    chemic attack kill five syrian aleppo sana aleppo
802       kill russian militari chopper shot syria idlib
803    syrian armi kill isil terrorist deir ezzor dei...
Name: content, Length: 804, dtype: object


In [19]:
X=data['content'].values
Y=data['labels'].values

converting textual to numerical data

In [20]:
vectorizer=TfidfVectorizer()
vectorizer.fit(X)
X=vectorizer.transform(X)

test and train

In [21]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,stratify=Y,random_state=1)

In [22]:
model=LogisticRegression()

In [23]:
model.fit(X_train,Y_train)

LogisticRegression()

In [24]:
X_train_pred=model.predict(X_train)
acc=accuracy_score(X_train_pred,Y_train)
print(acc)                                                          #acc of training data

0.7807153965785381


In [25]:
X_test_pred=model.predict(X_test)
acc=accuracy_score(X_test_pred,Y_test)
print(acc)                                                        #testing acc

0.5217391304347826


In [26]:
# Define the parameter grid for Logistic Regression
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100, 1000]
}

# Initialize GridSearchCV
grid_search = GridSearchCV(LogisticRegression(max_iter=1000, solver='liblinear'), param_grid, cv=5, scoring='accuracy', n_jobs=-1)

# Fit GridSearchCV to the training data
grid_search.fit(X_train, Y_train)

# Print the best parameters and best score
print("Best parameters: ", grid_search.best_params_)
print("Best cross-validation score: ", grid_search.best_score_)

# Get the best model
best_model = grid_search.best_estimator_
print(best_model)

Best parameters:  {'C': 0.1}
Best cross-validation score:  0.5303415697674418
LogisticRegression(C=0.1, max_iter=1000, solver='liblinear')


In [27]:
# Evaluate the best model on the training and test sets
X_train_pred_tuned = best_model.predict(X_train)
acc_train_tuned = accuracy_score(X_train_pred_tuned, Y_train)
print(f"Accuracy on training data after tuning: {acc_train_tuned}")

X_test_pred_tuned = best_model.predict(X_test)
acc_test_tuned = accuracy_score(X_test_pred_tuned, Y_test)
print(f"Accuracy on test data after tuning: {acc_test_tuned}")

Accuracy on training data after tuning: 0.578538102643857
Accuracy on test data after tuning: 0.5093167701863354
